# 업종별 경기를 비교할 수 있는 자료는 어디까지 있을까?

재무자료에 40개 업종이 있다고 해서 생산과 출하도 전부 관측되는 것은 아니다. 경기지표가 있는 곳부터 확인해야 다음 비교에서 어떤 업종이 빠지는지 알 수 있다.

ICR(이자보상배율)은 영업이익을 이자비용으로 나눈 값이다. 3배라면 영업이익이 그해 이자비용의 세 배라는 뜻이다. 한 행은 한 업종의 한 해다. 


[전체 질문 목록](../README.md)

**표 읽는 법**　N은 업종·연도 관측 수, G는 업종 수다. 회귀표의 beta는 추정한 관계의 크기, lo와 hi는 95% 신뢰구간의 양 끝이다. p값은 계수가 0이라는 가정에서 지금만큼 또는 더 극단적인 검정통계량이 나올 확률이다.

# 생산·출하 자료 수령 검증
2026-11-03. 원본 보존. CSV의 2행 헤더·코드·연도·원지수 단위 확인, 기존 40업종 코드 및 이름 대조, 결측과 전년 대비 로그변화 계산 가능성 검토. 회귀·표본 변경은 수행하지 않음. 이 폴더에서 Python(pandas,numpy)로 셀 실행. 코드 셀은 동일 Python 실행환경에서 실행하고 출력 저장. 분류 개정 이력·집계방법은 제공 메타자료로 확인 불가.

In [1]:
# 원지수의 연도와 중복 확인
from pathlib import Path
import pandas as pd,numpy as np,json,hashlib
BASE = Path.cwd()
(BASE / 'tables').mkdir(exist_ok=True)
for m in json.loads((BASE/'inputs/manifest.json').read_text(encoding='utf-8')):
 assert hashlib.sha256((BASE/'inputs'/m['file']).read_bytes()).hexdigest()==m['sha256']
d=pd.read_csv(BASE/'inputs/cycle.csv',header=[0,1],encoding='utf-8-sig')
b = pd.read_csv(BASE / 'inputs/base_panel.csv')
records = []
for _, r in d.iterrows():
    code, name = r.iloc[0].split(' ', 1)
    for j, c in enumerate(d.columns[1:], 1):
        year = int(c[0].split()[0][1:])
        item = c[1].split()[0]
        assert item in ['1', '2'] and '2020=100' in c[1]
        records.append(dict(code=code, industry=name, year=year, indicator={'1': 'production', '2': 'shipment'}[item], value=pd.to_numeric(r.iloc[j], errors='coerce'), raw=r.iloc[j]))
long = pd.DataFrame(records)
assert not long.duplicated(['code', 'year', 'indicator']).any()
assert set(long.year)==set(range(2017,2024))
assert (long.value.dropna()>0).all()

# 업종코드와 이름 대조
mapping=b[['code','industry']].drop_duplicates().merge(long[['code','industry']].drop_duplicates(),on='code',how='left',suffixes=('_panel','_cycle'),indicator=True)
assert (mapping.loc[mapping._merge=='both','industry_panel']==mapping.loc[mapping._merge=='both','industry_cycle']).all()

# 증가율 계산 및 결측 확인
wide=long.pivot(index=['code','industry','year'],columns='indicator',values='value').reset_index().sort_values(['code','year'])
for v in ['production','shipment']:
 wide[v+'_growth']=100*np.log(wide[v]/wide.groupby('code')[v].shift(1))
coverage=b[['code','industry','year']].merge(wide[['code','year','production_growth','shipment_growth']],on=['code','year'],how='left',validate='one_to_one')
for name,frame in [('source_long',long),('industry_mapping',mapping),('source_missing_cells',long[long.value.isna()]),('growth_series',wide),('panel_coverage',coverage),('unavailable_panel_rows',coverage[coverage.production_growth.isna()|coverage.shipment_growth.isna()])]:
 frame.to_csv(BASE/'tables'/f'{name}.csv',index=False,encoding='utf-8-sig')
print('Source rows:',len(d),'Existing industries:',b.code.nunique(),'Matching industries:',int((mapping._merge=='both').sum()))
print('Absent industries:',mapping[mapping._merge=='left_only'][['code','industry_panel']].to_dict('records'))
print('Missing source cells:',int(long.value.isna().sum()))
print('Matched available growth observations:',int(coverage[['production_growth','shipment_growth']].notna().all(axis=1).sum()),'/',len(b))
print(coverage[coverage.production_growth.isna()|coverage.shipment_growth.isna()].to_string(index=False))


Source rows: 43 Existing industries: 40 Matching industries: 38
Absent industries: [{'code': 'I3203', 'industry_panel': '기타 전자부품'}, {'code': 'I3305', 'industry_panel': '세라믹'}]
Missing source cells: 6
Matched available growth observations: 225 / 240
 code industry  year  production_growth  shipment_growth
I3203  기타 전자부품  2018                NaN              NaN
I3203  기타 전자부품  2019                NaN              NaN
I3203  기타 전자부품  2020                NaN              NaN
I3203  기타 전자부품  2021                NaN              NaN
I3203  기타 전자부품  2022                NaN              NaN
I3203  기타 전자부품  2023                NaN              NaN
I3209  기타 수송장비  2018                NaN              NaN
I3209  기타 수송장비  2019                NaN              NaN
I3209  기타 수송장비  2020                NaN              NaN
I3305      세라믹  2018                NaN              NaN
I3305      세라믹  2019                NaN              NaN
I3305      세라믹  2020                NaN              NaN
I3305     

### 출력에서 읽을 수 있는 것

Existing industries=40 중 Matching industries=38이다. 기타전자부품·세라믹 두 업종은 연결되지 않았고, 다른 일부 행도 전년도 값이 없어 증가율을 계산할 수 없었다. NaN은 결측이다.

- 원지수 자료를 업종·연도·지표별 한 행으로 정리했다. 2017~2023년을 포함하며 값이 있는 원지수는 모두 양수였다.
- 재무 패널과 연결되는 38개 업종에서 코드와 업종명이 일치했다. 기타전자부품과 세라믹은 원지수가 없었다.
- 생산과 출하의 전년도 대비 로그증가율을 계산하자 둘 다 확보되는 관측은 240행 중 225행이었다. 기타수송장비는 2018~2020년 증가율을 쓸 수 없었다.